# 🧠 Chain of Thought Prompting

**Welcome!** This notebook is your hands-on introduction to one of the most powerful ideas in prompt engineering: **Chain of Thought (CoT)** reasoning.

You'll discover that a single sentence added to a prompt can turn a confused, error-prone model into a careful, step-by-step problem solver. Pretty wild, right?

---

**How to use this notebook:**

- Cells marked **[RUN]** — just execute them, no changes needed.
- Cells marked **[TODO]** — you need to fill in some code before running.

> ⚠️ **Important:** Please select the **TORCH** kernel before starting.


## 1. Setup the environment and define utility functions

**[RUN]** The cells below install dependencies and load the model. No edits needed — just run them!


In [ ]:
# @title Install Dependencies {display-mode: "form"}
# @markdown Run this cell first to install the required packages.
!pip install transformers accelerate datasets

In [ ]:
# @title Load the Qwen Model {display-mode: "form"}
# @markdown Loads a small Qwen model for fast GPU inference.
import torch
import numpy as np
import random
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.float32,
)
model.eval()

In [ ]:
# @title Define the response generation logic {display-mode: "form"}

import uuid
import time
from IPython.display import display, HTML, Javascript
import html as html_lib


def generate_response(prompt, query_for_thinking=True):
    """
    Runs the model on a given prompt and returns the response text and elapsed time.
    """
    start = time.time()
    if query_for_thinking:
        prompt_inst = prompt.split("Question:")[0]
        prompt_q = "Output ONLY yes if this user prompt CONTAINS A REQUEST/AN INSTRUCTION to USE REASONING or THINK, otherwise output ONLY no. User prompt: " + prompt_inst
        resp, _ = generate_response(prompt_q, query_for_thinking=False)
        # print(f"cls response: {resp}")
        thinking_mode = "yes" in resp[-15:].lower()
    else:
        thinking_mode = True
    # print(f"{prompt=}, {thinking_mode=}, {query_for_thinking=}")
    inputs = tokenizer.apply_chat_template(
        [
            {
                "role": "system",
                "content": """Be concise in your answers. Clearly highlight what is your answer.""" ,
            },
            {"role": "user", "content": prompt},
        ],
        add_generation_prompt=True,
        enable_thinking=thinking_mode,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
        )
    response_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1] :],
        skip_special_tokens=True,
    )
    elapsed = time.time() - start
    return str(response_text), elapsed


In [ ]:
# @title Define Display Utilities {display-mode: "form"}
# @markdown Styled Markdown helpers for rendering prompts and model responses
from IPython.display import display, Markdown, clear_output

def display_sample(question, answer):
    display(Markdown(f"**Question:** {question}\n\n**Answer:** {answer}"))

def display_qwen(prompt):
    """
    Displays the prompt and model response using Markdown to properly render LaTeX.
    """
    print("⏳ Generating response...")

    # --- Call the model ---
    response_text, elapsed = generate_response(prompt)

    clear_output(wait=True)

    # Display using Markdown for LaTeX rendering
    md_content = f"""### Prompt:
{prompt}

### Model Response:
{response_text}

*⏱ Generated in {elapsed:.2f} seconds*"""
    display(Markdown(md_content))


## 2. Chain of Thought (CoT) Prompting

### What's the big idea?

Imagine asking someone a hard math question. If you just say _"Answer this"_, they might guess. But if you say _"Think through it step by step"_, suddenly they slow down, reason carefully, and are much more likely to get it right.

**Chain of Thought prompting does exactly this for LLMs.** By nudging the model to reason explicitly before answering, we unlock dramatically better performance on complex tasks — especially math, logic, and multi-step problems.

We'll test this on **MATH500**, a popular benchmark of competition-level math word problems. Spoiler: the difference will be very noticeable! 🚀


**[RUN]** Let's start by loading the MATH500 dataset and peeking at a sample question and its answer.


In [ ]:
train_split_main =  load_dataset("HuggingFaceH4/MATH-500", split="test")
print(f"Train split size: {len(train_split_main)}")

In [ ]:
IDX = 29
display_sample(train_split_main[IDX]["problem"], train_split_main[IDX]["solution"])

**[RUN]** Now let's ask our model to solve this question — no guidance, no hints, just the raw question (**zero-shot prompting**).


In [ ]:
question = train_split_main[IDX]["problem"]
prompt = f"""Question: {question}"""

display_qwen(prompt)

Hm, not great 😬. The model jumped straight to an answer without really thinking things through. This is what _pattern matching without reasoning_ looks like.

### ✏️ [TODO] Your turn — add a CoT trigger!

The fix is surprisingly simple. Complete the `cot_prompt` variable below with a short phrase that encourages the model to **think step by step** before answering.

> 💡 **Hint:** Think about how you'd ask a student to slow down and show their work. Even 5–6 words can be enough!


In [ ]:
cot_prompt = ""  # TODO: Add a Chain of Thought trigger phrase here
# --- 🎯🎯🎯🎯 ---
prompt = f"""{cot_prompt}, Question:
{question}"""

display_qwen(prompt)


🎉 **Amazing!** Such a tiny addition made such a big difference — the model is now reasoning step by step instead of just guessing.

This is the essence of Chain of Thought prompting: **words shape thinking**, even for AI systems.
